In [0]:
### This is an end-to-end coding playbook that covers real-time industry specific coding questions based on certains scenarios

In [0]:
### Initializing the saprk session 
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import*
spark=SparkSession.builder.appName("coding").getOrCreate()

In [0]:
#### QUESTION 1

### PROBLEM STATEMENT: ZenithCart is a fast-growing e-commerce company that sells home essentials across multiple Indian cities. The management team has noticed inconsistencies in their sales reports because customer, order, and return data are maintained across different operational systems. The Marketing team needs accurate return rates by product category for Tier-2 cities, while the Finance team wants visibility into customers with missing contact information. Additionally, leadership requires a daily revenue trend and a list of the company's highest-value customers to support strategic business decisions.

#### DATA COLLECTION: Every night, ZenithCart ingests data from three operational systems: Customer Management System (CMS), Order Management System (OMS), Returns Management System (RMS). Since these systems operate independently, some records contain missing values, duplicate entries, and unmatched relationships that must be handled during analysis.


zenith_orders= spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/1_zenithcart/1_zenithcart_datasets/orders (1).csv")

zenith_customers=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/1_zenithcart/1_zenithcart_datasets/customers.csv")

zenith_city_tier=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/1_zenithcart/1_zenithcart_datasets/city_tier.csv")

zenith_returns= spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/1_zenithcart/1_zenithcart_datasets/returns.csv")

In [0]:
## TASK 1: Calculate the Return Rate (%) for every Product Category considering only Tier-2 cities.

return_orders= zenith_orders.alias("or").join(zenith_returns.alias("re"), on=f.col("or.order_id").cast("string")==f.col("re.order_id").cast("string"),how="inner").select("or.*",f.col("re.return_id"),f.col("re.return_date"))

return_orders_customers= return_orders.alias("or").join(zenith_customers.alias("ct"),on=f.col("or.customer_id").cast("string")==f.col("ct.customer_id").cast("string"),how="left").select("or.*",f.col("ct.customer_name"),f.col("ct.city"))

return_orders_customers_city= return_orders_customers.alias("or").join(zenith_city_tier.alias("ct"),on=f.col("or.city")==f.col("ct.city"),how="left").select("or.*",f.col("ct.city_tier"))

### FINAL SOLUTION
total_orders_placed= return_orders.count()
required_df= return_orders_customers_city.filter(f.col("city_tier")==f.lit("Tier-2"))
grouped_df= return_orders_customers_city.groupBy(f.col("product_category")).agg(f.count("*").alias("total_return_orders"))
grouped_df= grouped_df.withColumn("return_percentage",f.round((f.col("total_return_orders")/total_orders_placed),3))

grouped_df.show()

+----------------+-------------------+-----------------+
|product_category|total_return_orders|return_percentage|
+----------------+-------------------+-----------------+
|         Kitchen|                  5|            0.333|
|        Cleaning|                  6|              0.4|
|       Furniture|                  2|            0.133|
|           Decor|                  2|            0.133|
+----------------+-------------------+-----------------+



In [0]:
## TASK 2: Find the number of customers having NULL email addresses grouped by signup channel.
null_email_channels= zenith_customers.filter(f.col("email").isNull())
final_null_email_channels= null_email_channels.groupBy(f.col("signup_channel")).agg(f.count("*").alias("total_null_emails")).orderBy(f.col("total_null_emails").desc())

final_null_email_channels.show()

+--------------+-----------------+
|signup_channel|total_null_emails|
+--------------+-----------------+
|      Referral|                3|
|       Website|                1|
|           App|                1|
+--------------+-----------------+



In [0]:
## TASK 3: Calculate the daily revenue trend considering only delivered orders.
delivered_orders= zenith_orders.filter(f.lower(f.col("order_status"))==f.lower(f.lit("Delivered")))
delivered_orders_revenue= delivered_orders.groupBy(f.col("order_date").cast("date")).agg(f.sum(f.col("order_amount")).alias("total_revenue")).orderBy(f.col("order_date").desc())
delivered_orders_revenue.show()

+----------+-------------+
|order_date|total_revenue|
+----------+-------------+
|2025-04-01|          535|
|2025-03-30|          766|
|2025-03-28|          668|
|2025-03-26|         3257|
|2025-03-24|         1711|
|2025-03-23|         2343|
|2025-03-19|         1653|
|2025-03-17|         2871|
|2025-03-15|         8759|
|2025-03-14|         2399|
|2025-03-13|         7164|
|2025-03-12|         3511|
|2025-03-11|         7179|
|2025-03-09|         1534|
|2025-03-08|         2756|
|2025-03-07|         4570|
|2025-03-03|          629|
|2025-02-24|         4891|
|2025-02-18|         1020|
|2025-02-17|          202|
+----------+-------------+
only showing top 20 rows


In [0]:
### TASK 4: Find the Top 10 customers based on their Net Spend.
## Net Spend = Sum(Delivered Order Amount) − Sum(Cancelled Order Amount)

customers_delivered_order_amount= zenith_orders.filter(f.col("order_status")==f.lit("Delivered")).groupBy(f.col("customer_id")).agg(f.sum(f.col("order_amount")).alias("total_delivered_order_amount"))

customers_returned_order_amount= zenith_orders.filter(f.col("order_status")==f.lit("Cancelled")).groupBy(f.col("customer_id")).agg(f.sum(f.col("order_amount")).alias("total_returned_order_amount"))

final_customer_spend_df= customers_delivered_order_amount.alias("a").join(customers_returned_order_amount.alias("b"), on=f.col("a.customer_id").cast("string")==f.col("b.customer_id").cast("string"),how="inner").select(f.col("a.customer_id"),f.col("a.total_delivered_order_amount"),f.col("b.total_returned_order_amount"))

final_customer_spend_df= final_customer_spend_df.withColumn("net_expenditure",f.round((f.col("total_delivered_order_amount")-f.col(("total_returned_order_amount"))),3))

### FINAL SOLUTION: Top 10 Customers
top_10_customers= final_customer_spend_df.orderBy(f.col("net_expenditure").desc()).limit(10)
top_10_customers.show()

+-----------+----------------------------+---------------------------+---------------+
|customer_id|total_delivered_order_amount|total_returned_order_amount|net_expenditure|
+-----------+----------------------------+---------------------------+---------------+
|       1030|                       11190|                       1761|           9429|
|       1032|                       10938|                       2599|           8339|
|       1010|                       16309|                       8820|           7489|
|       1021|                        7446|                        593|           6853|
|       1028|                        8260|                       4390|           3870|
|       1016|                        6080|                       2255|           3825|
|       1024|                        6176|                       2439|           3737|
|       1029|                        5097|                       2785|           2312|
|       1006|                        2646| 